# Marketing Mix Model with Meridian

Bayesian geo-level MMM that estimates the incremental ROAS of five paid channels and recommends how to
reallocate a fixed budget. Its outputs feed the Tableau dashboard *Marketing Budget Allocation Planner*.

Settings live in `src/config.py`. Comments on results refer to the reference run (`full` preset, seed 42).

## 1. Setup

In [ ]:
import shutil
import time
import warnings

import arviz as az
import matplotlib.pyplot as plt
import meridian
import numpy as np
import pandas as pd
from meridian.analysis import analyzer, optimizer
from meridian.analysis.review import reviewer
from meridian.data import load
from meridian.model import model, spec
from scipy import stats

from marketing_mix_model_dashboard.config import (
    ACTION_THRESHOLD_PCT, BREAKEVEN_BOUNDS, BUDGET_MULTIPLIERS, CHANNEL_FAMILY, CHANNEL_MAP,
    CONFIDENCE_LEVEL, CONTROL_COLS, CONTROL_RENAME, DATA_DICTIONARY_SRC, DATA_PATH, HOLDOUT_FRACTION,
    IMPRESSION_COLS, KNOTS, LOAD_SAVED_MODEL, MAX_LAG, MODEL_PATH, MODELS_DIR, NON_MEDIA_COLS,
    OPTIMIZER_DRAWS, ORGANIC_COLS, ORGANIC_MAP, PAID_CHANNELS, REPORTS_DIR, RUN_SENSITIVITY, SAMPLING,
    SAMPLING_PRESET, SEED, SENSITIVITY_PRESET, SENSITIVITY_VARIANTS, SPEND_BOUND, SPEND_COLS,
    TABLEAU_DIR, TRANCHE_PCT, slug,
)

for d in (TABLEAU_DIR, REPORTS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*trace group is not defined.*")
warnings.filterwarnings("ignore", message=".*set_random_seed is a no-op.*")
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

LO_Q, HI_Q = (1 - CONFIDENCE_LEVEL) / 2, 1 - (1 - CONFIDENCE_LEVEL) / 2
print("meridian", meridian.__version__, "| preset:", SAMPLING_PRESET, SAMPLING)


def gate(ok, warn=False):
    return "PASS" if ok else ("WARN" if warn else "FAIL")

## 2. Data
Google's simulated Meridian dataset (40 geos x 156 weeks), with channels renamed to fictional platforms.

In [ ]:
raw = pd.read_csv(DATA_PATH)
raw = raw.drop(columns=[c for c in raw.columns if c.startswith("Unnamed")])

rename = dict(CONTROL_RENAME)
for prefix, (name, _) in CHANNEL_MAP.items():
    rename[f"{prefix}_impression"] = f"{slug(name)}_impressions"
    rename[f"{prefix}_spend"] = f"{slug(name)}_spend"
for prefix, name in ORGANIC_MAP.items():
    rename[f"{prefix}_impression"] = slug(name)

df = raw.rename(columns=rename)
df["time"] = pd.to_datetime(df["time"]).dt.strftime("%Y-%m-%d")
df["revenue"] = df["conversions"] * df["revenue_per_conversion"]

N_GEOS, N_WEEKS = df["geo"].nunique(), df["time"].nunique()
YEARS = N_WEEKS / 52
print(f"{len(df):,} rows | {N_GEOS} geos x {N_WEEKS} weeks | {df['time'].min()} -> {df['time'].max()}")

numeric = IMPRESSION_COLS + SPEND_COLS + ORGANIC_COLS + ["conversions", "revenue_per_conversion", "population"]
pd.DataFrame({
    "check": ["No missing values", "Complete geo x week panel", "No duplicated rows",
              "No negative values", "Spend > 0 when impressions > 0"],
    "status": [
        gate(df.isna().sum().sum() == 0),
        gate(len(df) == N_GEOS * N_WEEKS),
        gate(not df.duplicated(["geo", "time"]).any()),
        gate((df[numeric] >= 0).all().all()),
        gate(all(((df[i] > 0) <= (df[s] > 0)).all() for i, s in zip(IMPRESSION_COLS, SPEND_COLS))),
    ],
})

## 3. Exploratory checks
Channels can only be separated if their spend varies and does not move in lockstep.

In [ ]:
weekly = df.groupby("time")[SPEND_COLS + ["revenue"]].sum()
weekly.index = pd.to_datetime(weekly.index)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
(weekly[SPEND_COLS] / 1e3).set_axis(PAID_CHANNELS, axis=1).plot(ax=axes[0], lw=1)
axes[0].set_title("Weekly national spend by channel (k)")
axes[0].legend(fontsize=8)
(weekly["revenue"] / 1e6).plot(ax=axes[1], color="black", lw=1)
axes[1].set_title("Weekly national revenue (M)")
plt.tight_layout()
plt.show()

In [ ]:
corr = df[SPEND_COLS].set_axis(PAID_CHANNELS, axis=1).corr()
spend_profile = pd.DataFrame({
    "spend_share_pct": 100 * df[SPEND_COLS].sum().values / df[SPEND_COLS].sum().sum(),
    "pct_weeks_zero_spend": [100 * (df[c] == 0).mean() for c in SPEND_COLS],
    "weekly_spend_cv": [weekly[c].std() / weekly[c].mean() for c in SPEND_COLS],
    "vif": np.diag(np.linalg.inv(corr.values)),
}, index=PAID_CHANNELS)
spend_profile

**Result:** all VIFs are below 2.5, so channels are separable. Instaview has zero spend in ~64% of
geo-weeks, which makes its estimates rely on fewer observations.

## 4. Model specification
Full spec: media, controls, Promo (non-media treatment) and organic media. 10% of geo-weeks are held out.

In [ ]:
def build_input_data(include_promo=True, include_organic=True):
    c2c = load.CoordToColumns(
        geo="geo", time="time", kpi="conversions", revenue_per_kpi="revenue_per_conversion",
        population="population", controls=CONTROL_COLS, media=IMPRESSION_COLS, media_spend=SPEND_COLS,
        non_media_treatments=NON_MEDIA_COLS if include_promo else None,
        organic_media=ORGANIC_COLS if include_organic else None,
    )
    return load.DataFrameDataLoader(
        df=df, coord_to_columns=c2c, kpi_type="non_revenue",
        media_to_channel=dict(zip(IMPRESSION_COLS, PAID_CHANNELS)),
        media_spend_to_channel=dict(zip(SPEND_COLS, PAID_CHANNELS)),
    ).load()


def build_spec(knots=KNOTS, holdout_id=None):
    kwargs = dict(media_prior_type="roi", max_lag=MAX_LAG, holdout_id=holdout_id)
    if knots == "aks":
        kwargs["enable_aks"] = True
    elif knots is not None:
        kwargs["knots"] = int(knots)
    return spec.ModelSpec(**kwargs)


input_data = build_input_data()
TIMES = [str(t)[:10] for t in input_data.time.values]
HOLDOUT_ID = np.random.default_rng(SEED).random((N_GEOS, N_WEEKS)) < HOLDOUT_FRACTION
mmm = model.Meridian(input_data=input_data, model_spec=build_spec(holdout_id=HOLDOUT_ID))
print(f"Knots: {mmm.knot_info.n_knots} | holdout cells: {HOLDOUT_ID.sum()} of {HOLDOUT_ID.size}")

**Knot choice:** with 66 (AKS) or 26 knots the time baseline did not converge (R-hat 1.8 / 1.6);
13 knots converged with the same holdout accuracy, so the simpler baseline was kept.

## 5. Posterior sampling

In [ ]:
if LOAD_SAVED_MODEL and MODEL_PATH.exists():
    mmm = model.load_mmm(str(MODEL_PATH))
    print("Loaded", MODEL_PATH)
else:
    t0 = time.time()
    mmm.sample_prior(SAMPLING["n_prior"], seed=SEED)
    mmm.sample_posterior(n_chains=SAMPLING["n_chains"], n_adapt=SAMPLING["n_adapt"],
                         n_burnin=SAMPLING["n_burnin"], n_keep=SAMPLING["n_keep"], seed=SEED)
    print(f"Sampling finished in {(time.time() - t0) / 60:.1f} min")
    model.save_mmm(mmm, str(MODEL_PATH))

mmm_analyzer = analyzer.Analyzer(model_context=mmm.model_context, inference_data=mmm.inference_data)

## 6. Convergence

In [ ]:
rhat = mmm_analyzer.rhat_summary()
posterior = mmm.inference_data.posterior
key_params = [p for p in ["roi_m", "beta_m", "alpha_m", "ec_m", "eta_m", "gamma_c", "gamma_n",
                          "beta_om", "alpha_om", "ec_om", "sigma", "knot_values"] if p in posterior]
diag = az.summary(mmm.inference_data, var_names=key_params, kind="diagnostics")
diag["param"] = diag.index.str.split("[").str[0]
roi_diag = diag[diag["param"] == "roi_m"].set_axis(PAID_CHANNELS)

divergent = np.asarray(mmm.inference_data.sample_stats["diverging"])
MAX_RHAT = float(rhat["max_r_hat"].max())
MIN_ESS_ROI = float(roi_diag["ess_bulk"].min())
MAX_RHAT_ROI = float(roi_diag["r_hat"].max())
PCT_DIV = float(divergent.mean())

convergence_gates = pd.DataFrame([
    ("Max R-hat (all parameters)", MAX_RHAT, "< 1.1 (WARN < 1.2)", gate(MAX_RHAT < 1.1, MAX_RHAT < 1.2)),
    ("Max R-hat (ROI parameters)", MAX_RHAT_ROI, "< 1.05", gate(MAX_RHAT_ROI < 1.05, MAX_RHAT_ROI < 1.1)),
    ("Min bulk ESS (ROI parameters)", MIN_ESS_ROI, "> 400 (WARN > 100)", gate(MIN_ESS_ROI > 400, MIN_ESS_ROI > 100)),
    ("Divergent transitions (%)", 100 * PCT_DIV, "< 1%", gate(PCT_DIV < 0.01, PCT_DIV < 0.05)),
], columns=["check", "value", "target", "status"])
convergence_gates

**Result:** max R-hat 1.03 and ROI ESS 528 (PASS). Divergences of 1.9% are a minor warning.

## 7. Goodness of fit
Out-of-sample quality is judged at geo level: national holdout metrics sum only ~4 geos per week.

In [ ]:
def predictive_accuracy_table(an):
    pa = an.predictive_accuracy().to_dataframe().reset_index()
    value_col = [c for c in pa.columns if c not in ("metric", "geo_granularity", "evaluation_set")][0]
    pa["geo_granularity"] = pa["geo_granularity"].str.capitalize()
    return pa.pivot_table(index=["geo_granularity", "metric"], columns="evaluation_set", values=value_col)


gof = predictive_accuracy_table(mmm_analyzer)
gof

In [ ]:
eva = mmm_analyzer.expected_vs_actual_data(aggregate_geos=True, confidence_level=CONFIDENCE_LEVEL)
fit_df = pd.DataFrame({
    "week": pd.to_datetime(TIMES),
    "actual": eva["actual"].values,
    "expected": eva["expected"].sel(metric="mean").values,
    "expected_ci_lo": eva["expected"].sel(metric="ci_lo").values,
    "expected_ci_hi": eva["expected"].sel(metric="ci_hi").values,
    "baseline": eva["baseline"].sel(metric="mean").values,
})

eva_geo = mmm_analyzer.expected_vs_actual_data(aggregate_geos=False)
pred = eva_geo["expected"].sel(metric="mean").transpose("geo", "time").values
act = eva_geo["actual"].transpose("geo", "time").values

fig, axes = plt.subplots(1, 3, figsize=(17, 4), gridspec_kw={"width_ratios": [2, 1, 1]})
axes[0].fill_between(fit_df["week"], fit_df["expected_ci_lo"], fit_df["expected_ci_hi"], alpha=0.25)
axes[0].plot(fit_df["week"], fit_df["actual"], color="black", lw=1, label="Actual")
axes[0].plot(fit_df["week"], fit_df["expected"], lw=1.2, label="Expected")
axes[0].plot(fit_df["week"], fit_df["baseline"], lw=1, ls="--", label="Baseline")
axes[0].set_title("National revenue")
axes[0].legend(fontsize=8)
for ax, mask, title in [(axes[1], ~HOLDOUT_ID, "Train"), (axes[2], HOLDOUT_ID, "Holdout")]:
    ax.scatter(act[mask], pred[mask], s=4, alpha=0.3)
    ax.plot([0, np.nanmax(act)], [0, np.nanmax(act)], color="red", lw=1)
    r2 = 1 - np.sum((act[mask] - pred[mask]) ** 2) / np.sum((act[mask] - act[mask].mean()) ** 2)
    ax.set_title(f"{title} geo-weeks (R² = {r2:.2f})")
plt.tight_layout()
plt.show()

**Result:** geo holdout R² 0.75 and wMAPE 0.21, only 0.9 pp above train (no overfitting).
National wMAPE is 3.9%.

## 8. Meridian model review

In [ ]:
review = reviewer.ModelReviewer(model_context=mmm.model_context, inference_data=mmm.inference_data).run()
review.output_model_health_card("model_health_card.html", str(REPORTS_DIR))
print(f"{review.overall_status.name} | health score {review.health_score:.1f}")
pd.Series(review.checks_status, name="status").to_frame()

**Result:** all 8 checks pass (health score 94).

## 9. Prior vs posterior
A posterior/prior SD ratio below 0.8 means the data, not the prior, drives the channel's ROI.

In [ ]:
prior_roi = np.asarray(mmm.inference_data.prior["roi_m"]).reshape(-1, len(PAID_CHANNELS))
post_roi = np.asarray(posterior["roi_m"]).reshape(-1, len(PAID_CHANNELS))
learning = pd.DataFrame({
    "channel": PAID_CHANNELS,
    "prior_median": np.median(prior_roi, 0),
    "posterior_median": np.median(post_roi, 0),
    "sd_ratio": post_roi.std(0) / prior_roi.std(0),
})
learning["data_informed"] = pd.cut(learning["sd_ratio"], [0, 0.5, 0.8, np.inf],
                                   labels=["Strong", "Moderate", "Weak"])

fig, axes = plt.subplots(1, len(PAID_CHANNELS), figsize=(16, 3))
bins = np.linspace(0, 6, 60)
for k, ax in enumerate(axes):
    ax.hist(prior_roi[:, k], bins=bins, density=True, alpha=0.4, label="Prior")
    ax.hist(post_roi[:, k], bins=bins, density=True, alpha=0.7, label="Posterior")
    ax.axvline(1, color="black", lw=0.8, ls=":")
    ax.set_title(PAID_CHANNELS[k], fontsize=9)
axes[0].legend(fontsize=7)
plt.show()
learning

## 10. ROAS, marginal ROAS and decision probabilities
ROAS = incremental revenue / spend. mROAS = return of the next unit of spend, the metric that ranks channels.

In [ ]:
roi_draws = np.asarray(mmm_analyzer.roi()).reshape(-1, len(PAID_CHANNELS))
mroi_draws = np.asarray(mmm_analyzer.marginal_roi()).reshape(-1, len(PAID_CHANNELS))
cpik_draws = np.asarray(mmm_analyzer.cpik()).reshape(-1, len(PAID_CHANNELS))
paid = mmm_analyzer.summary_metrics(confidence_level=CONFIDENCE_LEVEL).sel(
    distribution="posterior", channel=PAID_CHANNELS)

roi_table = pd.DataFrame({
    "channel": PAID_CHANNELS,
    "spend": paid["spend"].values,
    "incremental_revenue": paid["incremental_outcome"].sel(metric="mean").values,
    "incremental_revenue_ci_lo": paid["incremental_outcome"].sel(metric="ci_lo").values,
    "incremental_revenue_ci_hi": paid["incremental_outcome"].sel(metric="ci_hi").values,
    "contribution_pct": paid["pct_of_contribution"].sel(metric="mean").values,
    "roi_mean": roi_draws.mean(0),
    "roi_median": np.median(roi_draws, 0),
    "roi_ci_lo": np.quantile(roi_draws, LO_Q, 0),
    "roi_ci_hi": np.quantile(roi_draws, HI_Q, 0),
    "mroi_mean": mroi_draws.mean(0),
    "mroi_median": np.median(mroi_draws, 0),
    "mroi_ci_lo": np.quantile(mroi_draws, LO_Q, 0),
    "mroi_ci_hi": np.quantile(mroi_draws, HI_Q, 0),
    "cpik_median": np.median(cpik_draws, 0),
    "p_roi_gt_1": (roi_draws > 1).mean(0),
    "p_mroi_gt_1": (mroi_draws > 1).mean(0),
    "p_best_mroi": np.bincount(mroi_draws.argmax(1), minlength=len(PAID_CHANNELS)) / len(mroi_draws),
})
roi_table["spend_share_pct"] = 100 * roi_table["spend"] / roi_table["spend"].sum()

order = roi_table.sort_values("roi_median")["channel"].tolist()
t = roi_table.set_index("channel").loc[order]
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, key, title in [(axes[0], "roi", "ROAS"), (axes[1], "mroi", "Marginal ROAS")]:
    err = [t[f"{key}_median"] - t[f"{key}_ci_lo"], t[f"{key}_ci_hi"] - t[f"{key}_median"]]
    ax.errorbar(t[f"{key}_median"], range(len(order)), xerr=err, fmt="o", capsize=3)
    ax.axvline(1, color="red", ls="--", lw=1)
    ax.set_yticks(range(len(order)), order)
    ax.set_title(f"{title}: median and {int(CONFIDENCE_LEVEL * 100)}% CI")
plt.tight_layout()
plt.show()
roi_table.sort_values("mroi_median", ascending=False)

**Result:** Goobly (1.52) and Instaview (1.43) are the only channels with ROAS above 1. Every channel
returns less on the next unit (mROAS 0.34–0.80), so the budget case is reallocation, not expansion.

## 11. Sensitivity analysis
Alternative specs refit with lighter sampling, sharing the same holdout cells.

In [ ]:
def holdout_metric(an, metric="wMAPE", granularity="Geo"):
    t = predictive_accuracy_table(an)
    key = [k for k in t.index if k[0] == granularity and k[1] == metric]
    col = "Test" if "Test" in t.columns else t.columns[-1]
    return float(t.loc[key[0], col]) if key else np.nan


def fit_variant(name, knots, include_promo, include_organic, preset):
    m = model.Meridian(input_data=build_input_data(include_promo, include_organic),
                       model_spec=build_spec(knots, HOLDOUT_ID))
    if preset.get("n_prior"):
        m.sample_prior(preset["n_prior"], seed=SEED)
    m.sample_posterior(n_chains=preset["n_chains"], n_adapt=preset["n_adapt"],
                       n_burnin=preset["n_burnin"], n_keep=preset["n_keep"], seed=SEED)
    an = analyzer.Analyzer(model_context=m.model_context, inference_data=m.inference_data)
    r = np.asarray(an.roi()).reshape(-1, len(PAID_CHANNELS))
    mr = np.asarray(an.marginal_roi()).reshape(-1, len(PAID_CHANNELS))
    return pd.DataFrame({
        "spec": name, "channel": PAID_CHANNELS,
        "roi_median": np.median(r, 0), "roi_ci_lo": np.quantile(r, LO_Q, 0),
        "roi_ci_hi": np.quantile(r, HI_Q, 0), "mroi_median": np.median(mr, 0),
        "p_roi_gt_1": (r > 1).mean(0), "test_wmape_geo": holdout_metric(an),
        "max_r_hat": float(an.rhat_summary()["max_r_hat"].max()), "n_knots": m.knot_info.n_knots,
    })


sensitivity = [roi_table[["channel", "roi_median", "roi_ci_lo", "roi_ci_hi", "mroi_median", "p_roi_gt_1"]]
               .assign(spec="main", test_wmape_geo=holdout_metric(mmm_analyzer),
                       max_r_hat=MAX_RHAT, n_knots=mmm.knot_info.n_knots)]
if RUN_SENSITIVITY:
    for name, knots, promo, organic in SENSITIVITY_VARIANTS:
        t0 = time.time()
        sensitivity.append(fit_variant(name, knots, promo, organic, SENSITIVITY_PRESET))
        print(f"{name}: {(time.time() - t0) / 60:.1f} min")
sensitivity = pd.concat(sensitivity, ignore_index=True)

piv = sensitivity.pivot(index="channel", columns="spec", values="roi_median").loc[PAID_CHANNELS]
rank_corr = {s: round(float(stats.spearmanr(piv["main"], piv[s])[0]), 2) for s in piv.columns if s != "main"}
robustness = sensitivity.groupby("channel").agg(
    roi_min_across_specs=("roi_median", "min"),
    roi_max_across_specs=("roi_median", "max"),
    robust_vs_breakeven=("roi_median", lambda s: bool((s > 1).all() or (s < 1).all())),
).reindex(PAID_CHANNELS)

print("Rank correlation vs main:", rank_corr)
display(sensitivity.groupby("spec")[["test_wmape_geo", "max_r_hat", "n_knots"]].first())
piv.join(robustness)

**Result:** Goobly and Instaview are the top two channels in every spec. Displayr is the most
spec-sensitive channel (ROAS 0.84–1.40), and dropping Promo inflates LinkdUp, which confirms Promo belongs in the model.

## 12. Confidence tier per channel
One point each for: converged, ESS > 400, data-informed, decisive vs break-even and robust across specs.

In [ ]:
p1 = roi_table["p_roi_gt_1"].values
conf = pd.DataFrame({
    "channel": PAID_CHANNELS,
    "pt_converged": (roi_diag["r_hat"].values < 1.05).astype(int),
    "pt_ess": (roi_diag["ess_bulk"].values > 400).astype(int),
    "pt_data_informed": (learning["sd_ratio"].values < 0.8).astype(int),
    "pt_decisive": ((p1 >= 0.8) | (p1 <= 0.2)).astype(int),
})
if RUN_SENSITIVITY:
    conf["pt_robust"] = robustness["robust_vs_breakeven"].astype(int).values
points = [c for c in conf.columns if c.startswith("pt_")]
conf["confidence_score"] = conf[points].sum(1)
missing = len(points) - conf["confidence_score"]
conf["confidence_tier"] = np.select([missing == 0, missing == 1], ["High", "Medium"], "Low")
conf["roi_ci_relative_width"] = ((roi_table["roi_ci_hi"] - roi_table["roi_ci_lo"]) / roi_table["roi_median"]).values
conf

## 13. Response curves

In [ ]:
rc = mmm_analyzer.response_curves(spend_multipliers=[round(x, 2) for x in np.arange(0, 2.01, 0.05)],
                                  confidence_level=CONFIDENCE_LEVEL)
rc_df = pd.DataFrame([
    {
        "channel": ch,
        "spend_multiplier": float(m),
        "spend": float(rc["spend"].sel(channel=ch, spend_multiplier=m)),
        "incremental_revenue": float(rc["incremental_outcome"].sel(channel=ch, spend_multiplier=m, metric="mean")),
        "incremental_revenue_ci_lo": float(rc["incremental_outcome"].sel(channel=ch, spend_multiplier=m, metric="ci_lo")),
        "incremental_revenue_ci_hi": float(rc["incremental_outcome"].sel(channel=ch, spend_multiplier=m, metric="ci_hi")),
    }
    for ch in PAID_CHANNELS for m in rc.spend_multiplier.values
])
rc_df["roi_at_point"] = np.where(rc_df["spend"] > 0, rc_df["incremental_revenue"] / rc_df["spend"], np.nan)

fig, axes = plt.subplots(1, len(PAID_CHANNELS), figsize=(17, 3.2))
for ax, ch in zip(axes, PAID_CHANNELS):
    d = rc_df[rc_df["channel"] == ch]
    ax.fill_between(d["spend"] / 1e6, d["incremental_revenue_ci_lo"] / 1e6,
                    d["incremental_revenue_ci_hi"] / 1e6, alpha=0.25)
    ax.plot(d["spend"] / 1e6, d["incremental_revenue"] / 1e6)
    cur = d[d["spend_multiplier"] == 1.0]
    ax.scatter(cur["spend"] / 1e6, cur["incremental_revenue"] / 1e6, color="black", zorder=3, s=15)
    ax.plot([0, d["spend"].max() / 1e6], [0, d["spend"].max() / 1e6], color="grey", ls=":", lw=0.8)
    ax.set_title(ch, fontsize=9)
    ax.set_xlabel("Spend (M)")
axes[0].set_ylabel("Incremental revenue (M)")
plt.tight_layout()
plt.show()

## 14. Budget optimization
The optimizer runs on a thinned posterior (`OPTIMIZER_DRAWS`): runtime scales with draws, the allocation does not.

In [ ]:
n_chain, n_draw = posterior.sizes["chain"], posterior.sizes["draw"]
thin_step = max(1, (n_chain * n_draw) // OPTIMIZER_DRAWS)
opt_analyzer = analyzer.Analyzer(
    model_context=mmm.model_context,
    inference_data=az.InferenceData(posterior=posterior.sel(draw=slice(None, None, thin_step)),
                                    prior=mmm.inference_data.prior),
)
budget_optimizer = optimizer.BudgetOptimizer(analyzer=opt_analyzer)
HIST_BUDGET = float(df[SPEND_COLS].sum().sum())
print(f"Optimizer uses {n_chain * len(range(0, n_draw, thin_step))} of {n_chain * n_draw} draws")


def opt_frame(res, scenario):
    rows = []
    for label, ds in (("Current", res.nonoptimized_data), ("Optimized", res.optimized_data)):
        for ch in PAID_CHANNELS:
            rows.append({
                "scenario": scenario, "allocation": label, "channel": ch,
                "spend": float(ds["spend"].sel(channel=ch)),
                "incremental_revenue": float(ds["incremental_outcome"].sel(channel=ch, metric="mean")),
                "incremental_revenue_ci_lo": float(ds["incremental_outcome"].sel(channel=ch, metric="ci_lo")),
                "incremental_revenue_ci_hi": float(ds["incremental_outcome"].sel(channel=ch, metric="ci_hi")),
                "roi": float(ds["roi"].sel(channel=ch, metric="mean")),
                "mroi": float(ds["mroi"].sel(channel=ch, metric="mean")),
                "total_budget": float(ds.attrs["budget"]),
                "total_incremental_revenue": float(ds.attrs["total_incremental_outcome"]),
                "total_roi": float(ds.attrs["total_roi"]),
            })
    return pd.DataFrame(rows)


def optimize_fixed(budget=None):
    return budget_optimizer.optimize(fixed_budget=True, budget=budget, spend_constraint_lower=SPEND_BOUND,
                                     spend_constraint_upper=SPEND_BOUND, confidence_level=CONFIDENCE_LEVEL)


scenario_results = {mult: optimize_fixed(HIST_BUDGET * mult) for mult in BUDGET_MULTIPLIERS}
scenarios = pd.concat([opt_frame(res, f"Budget {int(m * 100)}%").assign(budget_multiplier=m)
                       for m, res in scenario_results.items()], ignore_index=True)

base_result = scenario_results[1.0]
base_result.output_optimization_summary("budget_optimization_report.html", str(REPORTS_DIR))
allocation = opt_frame(base_result, "Fixed budget (100%)")

alloc_wide = allocation.pivot(index="channel", columns="allocation", values="spend").loc[PAID_CHANNELS]
alloc_wide["change"] = alloc_wide["Optimized"] - alloc_wide["Current"]
alloc_wide["change_pct"] = 100 * alloc_wide["change"] / alloc_wide["Current"]
tot = allocation.groupby("allocation")[["total_incremental_revenue", "total_roi"]].first()
uplift = tot.loc["Optimized", "total_incremental_revenue"] - tot.loc["Current", "total_incremental_revenue"]
print(f"Uplift: {uplift / YEARS:,.0f} per year ({100 * uplift / tot.loc['Current', 'total_incremental_revenue']:.1f}%)")

display(scenarios.pivot_table(index="budget_multiplier", columns="allocation",
                              values=["total_incremental_revenue", "total_roi"], aggfunc="first"))
alloc_wide

**Result:** at the same budget, spend moves from Displayr (−25%) and LinkdUp (−8%) to Goobly, Instaview
(+30%, the bound) and Facepage (+9%), for about +1.2M revenue per year (+1.5%).

In [ ]:
lower, upper = BREAKEVEN_BOUNDS
breakeven_df = opt_frame(budget_optimizer.optimize(
    fixed_budget=False, target_mroi=1.0, spend_constraint_lower=lower,
    spend_constraint_upper=upper, confidence_level=CONFIDENCE_LEVEL,
), "Break-even (target mROI = 1)")
be_opt = breakeven_df[breakeven_df["allocation"] == "Optimized"].set_index("channel")
be_cur = breakeven_df[breakeven_df["allocation"] == "Current"].set_index("channel")
BE_TOTAL = float(be_opt["total_budget"].iloc[0])
BE_BOUND_BINDING = bool(((be_opt["spend"] <= (1 - lower) * be_cur["spend"] * 1.01) & (be_opt["mroi"] < 0.99)).any())
print(f"Break-even budget: {BE_TOTAL:,.0f} ({100 * (BE_TOTAL / HIST_BUDGET - 1):+.1f}% vs historical)"
      + (" | upper estimate, lower bound binding" if BE_BOUND_BINDING else ""))

## 15. Spending tranches
mROAS describes a 1% change; these columns give the return of a ±20% tranche and of the recommended move.

In [ ]:
spend_vec = roi_table.set_index("channel").loc[PAID_CHANNELS, "spend"].values
tag = int(TRANCHE_PCT * 100)


def tranche_draws(sf0, sf1):
    x = np.asarray(mmm_analyzer.incremental_outcome(scaling_factor0=sf0, scaling_factor1=sf1,
                                                    include_non_paid_channels=False))
    return x.reshape(-1, x.shape[-1])


def summarize(prefix, values):
    return {prefix: np.median(values, 0),
            f"{prefix}_ci_lo": np.quantile(values, LO_Q, 0),
            f"{prefix}_ci_hi": np.quantile(values, HI_Q, 0)}


up = tranche_draws(1.0, 1.0 + TRANCHE_PCT)
down = tranche_draws(1.0 - TRANCHE_PCT, 1.0)
tranche_spend = TRANCHE_PCT * spend_vec

move_ratio = (alloc_wide["Optimized"] / alloc_wide["Current"]).loc[PAID_CHANNELS].values
move_rev = np.zeros_like(up)
for k, r in enumerate(move_ratio):
    if abs(r - 1) > 1e-9:
        move_rev[:, k] = (tranche_draws(1.0, r) if r > 1 else -tranche_draws(r, 1.0))[:, k]
move_spend = (move_ratio - 1) * spend_vec

tranche_table = pd.DataFrame({
    "channel": PAID_CHANNELS,
    "return_per_100_next_unit": 100 * roi_table["mroi_median"].values,
    f"plus{tag}_extra_spend": tranche_spend,
    **summarize(f"plus{tag}_extra_revenue", up),
    **summarize(f"plus{tag}_return_per_100", 100 * up / tranche_spend),
    f"minus{tag}_spend_saved": tranche_spend,
    **summarize(f"minus{tag}_revenue_lost", down),
    **summarize(f"minus{tag}_revenue_lost_per_100", 100 * down / tranche_spend),
    "recommended_spend_change": move_spend,
    **summarize("recommended_revenue_change", move_rev),
})
with np.errstate(divide="ignore", invalid="ignore"):
    tranche_table["recommended_return_per_100"] = np.where(
        np.abs(move_spend) > 0, 100 * np.median(move_rev, 0) / np.abs(move_spend), np.nan)

tranche_table[["channel", "return_per_100_next_unit", f"plus{tag}_return_per_100",
               f"minus{tag}_revenue_lost_per_100", "recommended_return_per_100"]].round(1)

## 16. Channel scorecard

In [ ]:
def recommendation(action, tier, worth_new_budget):
    if action == "Maintain":
        return "Keep current spend"
    if action == "Increase":
        base = "Increase spend (worth new budget)" if worth_new_budget else "Receive budget from lower-return channels"
    else:
        base = "Release budget to higher-return channels"
    if tier == "High":
        return base
    return f"{base} - phase in and monitor" if tier == "Medium" else f"{base} - validate with a geo-lift test first"


scorecard = (
    roi_table
    .merge(conf[["channel", "confidence_score", "confidence_tier", "roi_ci_relative_width"]], on="channel")
    .merge(learning[["channel", "sd_ratio", "data_informed"]], on="channel")
    .merge(robustness.reset_index(), on="channel")
    .merge(alloc_wide[["Optimized", "change", "change_pct"]].reset_index().rename(columns={
        "Optimized": "optimized_spend", "change": "spend_change", "change_pct": "spend_change_pct"}), on="channel")
    .merge(tranche_table, on="channel")
)
scorecard.insert(1, "channel_family", scorecard["channel"].map(CHANNEL_FAMILY))
scorecard["roi_r_hat"] = roi_diag["r_hat"].values
scorecard["roi_ess_bulk"] = roi_diag["ess_bulk"].values
scorecard["annual_spend"] = scorecard["spend"] / YEARS
scorecard["annual_optimized_spend"] = scorecard["optimized_spend"] / YEARS
scorecard["action"] = np.select(
    [scorecard["spend_change_pct"] > ACTION_THRESHOLD_PCT, scorecard["spend_change_pct"] < -ACTION_THRESHOLD_PCT],
    ["Increase", "Decrease"], "Maintain")
scorecard["worth_new_budget"] = scorecard["p_mroi_gt_1"] > 0.5
scorecard["recommendation"] = [recommendation(a, t, w) for a, t, w in
                               zip(scorecard["action"], scorecard["confidence_tier"], scorecard["worth_new_budget"])]
scorecard["priority_rank"] = scorecard["mroi_median"].rank(ascending=False).astype(int)
scorecard = scorecard.sort_values("priority_rank")
scorecard[["priority_rank", "channel", "roi_median", "return_per_100_next_unit",
           "spend_change_pct", "confidence_tier", "recommendation"]]

## 17. Exports for Tableau

In [ ]:
def export(frame, name):
    frame.to_csv(TABLEAU_DIR / name, index=False)
    print(f"{name:<34} {len(frame):>6} rows")


def gof_val(granularity, metric, col):
    flat = gof.reset_index()
    row = flat[(flat["geo_granularity"] == granularity) & (flat["metric"].str.lower() == metric.lower())]
    return float(row[col].iloc[0]) if len(row) and col in row else np.nan


geo_wmape_train, geo_wmape_test = gof_val("Geo", "wMAPE", "Train"), gof_val("Geo", "wMAPE", "Test")
quality_rows = [("Convergence", *r) for r in convergence_gates.itertuples(index=False)] + [
    ("Fit", "National R² (all data)", gof_val("National", "R_Squared", "All Data"), "> 0.5", None),
    ("Fit", "National wMAPE (all data)", gof_val("National", "wMAPE", "All Data"), "< 0.05", None),
    ("Fit", "Geo R² (holdout)", gof_val("Geo", "R_Squared", "Test"), "> 0.7", None),
    ("Fit", "Geo wMAPE (holdout)", geo_wmape_test, "< 0.25", None),
    ("Fit", "Overfitting gap: geo wMAPE holdout - train", geo_wmape_test - geo_wmape_train, "< 0.03", None),
    ("Review", "Meridian review overall status", None, "PASS", review.overall_status.name),
    ("Review", "Meridian health score", review.health_score, "> 70", None),
] + [("Review", f"Check: {k}", None, "PASS", v) for k, v in review.checks_status.items()]
model_quality = pd.DataFrame(quality_rows, columns=["category", "check", "value", "target", "status"])
model_quality["status"] = [
    s if s is not None or pd.isna(v)
    else gate(v > float(tg.split()[1]) if tg.startswith(">") else v < float(tg.split()[1]))
    for s, v, tg in zip(model_quality["status"], model_quality["value"], model_quality["target"])
]
model_quality["model_run"] = f"{SAMPLING_PRESET} | knots={mmm.knot_info.n_knots} | seed={SEED}"

keep = np.linspace(0, len(roi_draws) - 1, min(len(roi_draws), 2000)).astype(int)
draws = pd.concat([pd.DataFrame({"channel": ch, "draw_id": np.arange(len(keep)),
                                 "roi": roi_draws[keep, k], "mroi": mroi_draws[keep, k]})
                   for k, ch in enumerate(PAID_CHANNELS)])

inc_t = np.asarray(mmm_analyzer.incremental_outcome(aggregate_times=False, include_non_paid_channels=False))
inc_t = inc_t.reshape(-1, inc_t.shape[-2], inc_t.shape[-1]).mean(0)
spend_t = np.asarray(input_data.media_spend.sum("geo").transpose("time", "media_channel"))
weekly_contrib = pd.DataFrame([{"week": TIMES[t], "channel": ch, "incremental_revenue": inc_t[t, k],
                                "spend": spend_t[t, k]}
                               for t in range(len(TIMES)) for k, ch in enumerate(PAID_CHANNELS)])

cur_tot, opt_tot = tot.loc["Current"], tot.loc["Optimized"]
best = scorecard.sort_values("p_best_mroi", ascending=False).iloc[0]
dashboard_kpis = pd.DataFrame([{
    "period_start": TIMES[0], "period_end": TIMES[-1], "period_years": round(YEARS, 2),
    "historical_budget": HIST_BUDGET,
    "annual_historical_budget": HIST_BUDGET / YEARS,
    "current_incremental_revenue": cur_tot["total_incremental_revenue"],
    "optimized_incremental_revenue": opt_tot["total_incremental_revenue"],
    "reallocation_uplift": uplift,
    "reallocation_uplift_pct": 100 * uplift / cur_tot["total_incremental_revenue"],
    "annual_reallocation_uplift": uplift / YEARS,
    "current_roi": cur_tot["total_roi"],
    "optimized_roi": opt_tot["total_roi"],
    "best_channel_next_unit": best["channel"],
    "best_channel_p_best": best["p_best_mroi"],
    "channels_worth_new_budget": int(scorecard["worth_new_budget"].sum()),
    "breakeven_budget": BE_TOTAL,
    "spend_above_breakeven_pct": max(0.0, 100 * (1 - BE_TOTAL / HIST_BUDGET)),
    "breakeven_is_upper_estimate": BE_BOUND_BINDING,
    "model_review_status": review.overall_status.name,
    "model_health_score": review.health_score,
}])

rc_export = rc_df.merge(alloc_wide[["Current", "Optimized"]].reset_index()
                        .rename(columns={"Current": "current_spend", "Optimized": "optimized_spend"}), on="channel")

export(dashboard_kpis, "dashboard_kpis.csv")
export(scorecard.round(6), "channel_scorecard.csv")
export(allocation.assign(channel_family=allocation["channel"].map(CHANNEL_FAMILY)), "budget_allocation.csv")
export(scenarios.assign(channel_family=scenarios["channel"].map(CHANNEL_FAMILY)), "budget_scenarios.csv")
export(rc_export, "response_curves.csv")
export(draws, "roi_posterior_draws.csv")
export(fit_df.assign(week=fit_df["week"].dt.strftime("%Y-%m-%d")), "weekly_model_fit.csv")
export(weekly_contrib, "weekly_channel_contribution.csv")
export(model_quality, "model_quality.csv")
export(sensitivity, "sensitivity_roi.csv")
shutil.copy(DATA_DICTIONARY_SRC, TABLEAU_DIR / "DATA_DICTIONARY.md")

## 18. Limitations
- ROAS priors are generic: geo-lift experiments on low-confidence channels would be the strongest upgrade.
- The ±30% optimizer bound keeps recommendations within the observed spend range.
- The data is simulated, so results illustrate the method rather than a real business.